In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

In [7]:
from rag.retrieval import retrieval
from rag.vector_database import VectorDatabase
from rag.utils import init_reranker
import pandas as pd


c:\Users\hafsa\Desktop\Islamic-AI-Assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:21<00:00, 17.95it/s]


In [35]:
data_path = "../../data/evaluation/eval_dataset_batch2.csv"
doc_path ="../../data/gold"
data = pd.read_csv(data_path)
vector_db = VectorDatabase(
        collection_name="quran_vdb",
        path=doc_path,
    )

Loading weights: 100%|██████████| 391/391 [00:04<00:00, 88.77it/s]


In [ ]:
reranker = init_reranker()
data = data.assign(
    parent_id_gt=lambda df_: (
        df_["surah_n"].astype(str)
        + ":"
        + df_["ayah_n"].astype(str)
    )
)
k_param =  [1, 
            3, 5, 7, 10, 15]

for k in k_param:
    col_name = f"result_{k}"
    col_check = f"check_{k}"

    # create empty column
    data[col_name] = ""

    for i, row in data.iterrows():

        results, parent_ids = retrieval(
            docs_path=os.path.join(doc_path, "quran_docs.pkl"),
            query=row["question"],
            vector_db=vector_db,
            top_k=k,
            return_parent_ids=True,
            reranker=reranker
        )
        data.loc[i,col_check] = data.loc[i,"parent_id_gt"] in parent_ids
        data.loc[i, col_name] = ",".join(parent_ids)



In [ ]:
data.to_csv("../../data/evaluation/eval_data_final_after_reranking.csv",index=False)

### Read the data

In [ ]:
import plotly.express as px
import pandas as pd
data_after = pd.read_csv("../../data/evaluation/eval_data_final_after_reranking.csv")
data_before = pd.read_csv("../../data/evaluation/eval_data_final.csv")

### Calculate metrics: Recall@K

In [ ]:
def recall_calcul(data):
    rows = []

    for col in data.columns:
        if col.startswith("check"):
            rows.append([
                col.split('_')[1],
                data[col].sum() / len(data)
            ])

    data_benchmark = pd.DataFrame(rows, columns=['value_k', 'value'])
    data["recall_1"] = data["check_1"].astype(int)
    data["recall_3"] = data["check_3"].astype(int)
    data["recall_5"] = data["check_5"].astype(int)
    data["recall_7"] = data["check_7"].astype(int)
    data["recall_10"] = data["check_10"].astype(int)
    data["recall_15"] = data["check_15"].astype(int)
    return data

data_before=recall_calcul(data_before)
data_after=recall_calcul(data_after)

In [ ]:

recall_cols = ["recall_1","recall_3","recall_5","recall_7","recall_10","recall_15"]

mean_scores_before = data_before[recall_cols].mean()
mean_scores_after = data_after[recall_cols].mean()

df_plot_before = pd.DataFrame({
    "K": [1, 3, 5, 7, 10, 15],
    "Recall Before": mean_scores_before.values*100
})
fig = px.line(
    df_plot_before,
    x="K",
    y="Recall Before",
    markers=True,
    title="Retrieval Performance: Recall@K (%) vs K",
)

fig.update_layout(
    xaxis_title="K (Top-K retrieved documents)",
    yaxis_title="Recall@K (%)",
    template="simple_white"
)


fig.show()

### Calculate metrics: MRR

In [ ]:
def mrr_calcul(data):
    for col in data.columns:

        if col.startswith("result_"):

            k = col.split("_")[1]
            mrr_col = f"mrr_{k}"

            def compute_mrr(row):

                gt = row["parent_id_gt"]

                retrieved = row[col]

                # handle empty retrieval
                if pd.isna(retrieved) or retrieved == "":
                    return 0.0

                retrieved_list = retrieved.split(",")

                # search for first correct occurrence
                for rank, pid in enumerate(retrieved_list, start=1):

                    if pid.strip() == gt:
                        return 1 / rank

                return 0.0

            data[mrr_col] = data.apply(compute_mrr, axis=1)

    return data

data_before = mrr_calcul(data_before)
data_after = mrr_calcul(data_after)

In [ ]:

mrr_cols = ["mrr_1","mrr_3","mrr_5","mrr_7","mrr_10","mrr_15"]

mean_scores = data_before[mrr_cols].mean()

df_plot = pd.DataFrame({
    "K": [1, 3, 5, 7, 10, 15],
    "mrr": mean_scores.values*100
})
fig = px.line(
    df_plot,
    x="K",
    y="mrr",
    markers=True,
    title="Retrieval Performance: mrr@K (%) vs K",
)

fig.update_layout(
    xaxis_title="K (Top-K retrieved documents)",
    yaxis_title="mrr@K (%)",
    template="simple_white"
)

fig.show()

In [ ]:
import plotly.graph_objects as go

k_values = [1, 3, 5, 7, 10, 15]

# ---------------------------------------------------
# Recall
# ---------------------------------------------------
recall_cols = [
    "recall_1",
    "recall_3",
    "recall_5",
    "recall_7",
    "recall_10",
    "recall_15"
]

recall_scores = data[recall_cols].mean().values

# ---------------------------------------------------
# MRR
# ---------------------------------------------------
mrr_cols = [
    "mrr_1",
    "mrr_3",
    "mrr_5",
    "mrr_7",
    "mrr_10",
    "mrr_15"
]

mrr_scores = data[mrr_cols].mean().values

# ---------------------------------------------------
# Figure
# ---------------------------------------------------
fig = go.Figure()

# Recall curve
fig.add_trace(
    go.Scatter(
        x=k_values,
        y=recall_scores,
        mode="lines+markers",
        name="Recall@K"
    )
)

# MRR curve
fig.add_trace(
    go.Scatter(
        x=k_values,
        y=mrr_scores,
        mode="lines+markers",
        name="MRR@K"
    )
)

# ---------------------------------------------------
# Layout
# ---------------------------------------------------
fig.update_layout(
    title="Retrieval Performance vs K",
    xaxis_title="K (Top-K Retrieved Documents)",
    yaxis_title="Score",
    template="simple_white",
    legend_title="Metric",
)

fig.update_yaxes()

fig.show()

In [ ]:
import plotly.graph_objects as go

k_values = [1, 3, 5, 7, 10, 15]

# ===================================================
# BEFORE
# ===================================================

recall_before = data_before[
    [
        "recall_1",
        "recall_3",
        "recall_5",
        "recall_7",
        "recall_10",
        "recall_15",
    ]
].mean().values

mrr_before = data_before[
    [
        "mrr_1",
        "mrr_3",
        "mrr_5",
        "mrr_7",
        "mrr_10",
        "mrr_15",
    ]
].mean().values

# ===================================================
# AFTER
# ===================================================

recall_after = data_after[
    [
        "recall_1",
        "recall_3",
        "recall_5",
        "recall_7",
        "recall_10",
        "recall_15",
    ]
].mean().values

mrr_after = data_after[
    [
        "mrr_1",
        "mrr_3",
        "mrr_5",
        "mrr_7",
        "mrr_10",
        "mrr_15",
    ]
].mean().values

# ===================================================
# FIGURE
# ===================================================

fig = go.Figure()

# ---------------------------------------------------
# Recall BEFORE
# ---------------------------------------------------
fig.add_trace(
    go.Scatter(
        x=k_values,
        y=recall_before,
        mode="lines+markers",
        name="Recall@K (Before)"
    )
)


# ---------------------------------------------------
# MRR BEFORE
# ---------------------------------------------------
fig.add_trace(
    go.Scatter(
        x=k_values,
        y=mrr_before,
        mode="lines+markers",
        name="MRR@K (Before)"
    )
)

# ---------------------------------------------------
# MRR AFTER
# ---------------------------------------------------
fig.add_trace(
    go.Scatter(
        x=k_values,
        y=mrr_after,
        mode="lines+markers",
        name="MRR@K (After)"
    )
)
# ---------------------------------------------------
# Recall AFTER
# ---------------------------------------------------
fig.add_trace(
    go.Scatter(
        x=k_values,
        y=recall_after,
        mode="markers",
        name="Recall@K (After)"
    )
)

# ===================================================
# LAYOUT
# ===================================================

fig.update_layout(
    title="Retrieval Performance Before vs After Reranking",
    xaxis_title="K (Top-K Retrieved Chunks)",
    yaxis_title="Score",
    template="simple_white",
    legend_title="Metrics",

)

fig.show()

## Generative

In [8]:
from agents.quran_mufasir import QuranRAG
agent = QuranRAG(
    docs_path="../../data/gold/quran_docs.pkl",
    vector_db_path="../../data/gold"

)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 3038.00it/s]


In [9]:
import pandas as pd
data_generator_eval = pd.read_json("../../data/evaluation/eval_generator_dataset.json")

In [10]:
import re

def parse_agent_response(text):
    result = {
        "generated_answer": "",
        "generated_surah_ayah": None,
        "generated_reasoning": None,
    }

    # Extract reference
    ref_match = re.search(
        r"Reference:\s*Surah\s*(\d+),\s*Ayah\s*(\d+)",
        text,
        re.IGNORECASE
    )

    if ref_match:
        surah = ref_match.group(1)
        ayah = ref_match.group(2)
        result["generated_surah_ayah"] = f"{surah}:{ayah}"

    # Extract tafsir insight
    insight_match = re.search(
        r"Tafsir Insights:\s*(.*)",
        text,
        re.IGNORECASE | re.DOTALL
    )

    if insight_match:
        result["generated_reasoning"] = insight_match.group(1).strip()

    # Main answer = everything before "Reference:"
    answer_split = text.split("Reference:")
    result["generated_answer"] = answer_split[0].strip()

    return result

In [11]:
import os
import json
import pandas as pd
from tqdm import tqdm

output_path = "../../data/evaluation/eval_generator_dataset_full.jsonl"

# Load already processed questions if file exists
processed_questions = set()

if os.path.exists(output_path):
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            processed_questions.add(item["question"])

print(f"Already processed: {len(processed_questions)}")

for i, row in tqdm(data_generator_eval.iterrows(), total=len(data_generator_eval)):

    question = row["question"]

    # Skip already processed rows
    if question in processed_questions:
        continue

    try:
        # Raw agent response
        raw_response = agent.ask(question=question)

        # Parse response
        parsed_response = parse_agent_response(raw_response)

        result = {
            # Question
            "question": question,

            # Ground Truth
            "ground_truth_answer": row["answer"],
            "ground_truth_surah_ayah": row["surah_ayah"],
            "ground_truth_reasoning": row["reasoning_span"],

            # Agent Output
            "agent_raw_response": raw_response,
            "generated_answer": parsed_response["generated_answer"],
            "generated_surah_ayah": parsed_response["generated_surah_ayah"],
            "generated_reasoning": parsed_response["generated_reasoning"],
        }

        # Append immediately to file
        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    except Exception as e:
        print(f"Error at row {i}: {e}")

Already processed: 0


  0%|          | 0/73 [00:00<?, ?it/s]

STEP 1: RETRIEVING CONTEXT
STEP 1: Loading QuranDoc store
Docs path: ../../data/gold/quran_docs.pkl
Successfully loaded 10 QuranDoc objects.

STEP 2: Dense semantic retrieval
Query: What did the hypocrites say when a calamity overtook the Prophet?
Top-K retrieval: 1
Vector search completed successfully.

STEP 3: Building retrieved chunk candidates
[1]
Chunk ID      : 9:50:0
Parent ID     : 9:50
Distance      : 0.4146


STEP 4: Cross-encoder reranking
Reranker detected.
Computing rerank scores...
Chunk 9:50:0 | Original Rank: 1 | Rerank Score: 0.9856

Reranking completed.

NEW ORDER AFTER RERANKING:
--------------------------------------------------------------------------------
[NEW RANK 1] 9:50:0 | Original Rank=1 | Score=0.9856

STEP 5: Diversity filtering
Chunks after diversity filtering: 1

Kept chunk 9:50:0 | Parent=9:50

STEP 6: Selecting final chunks
Final chunk count: 1

[FINAL 1]
Chunk ID      : 9:50:0
Parent ID     : 9:50
Original Rank : 1
Rerank Score  : 0.9856

STEP 7: Buil

  1%|▏         | 1/73 [01:14<1:29:00, 74.17s/it]


DONE
GENERATION COMPLETED
STEP 1: RETRIEVING CONTEXT
STEP 1: Loading QuranDoc store
Docs path: ../../data/gold/quran_docs.pkl
Successfully loaded 10 QuranDoc objects.

STEP 2: Dense semantic retrieval
Query: According to the tafsir of Surah 9:50, what does the Prophet say in reply to the hypocrites' enmity?
Top-K retrieval: 1
Vector search completed successfully.

STEP 3: Building retrieved chunk candidates
[1]
Chunk ID      : 9:50:0
Parent ID     : 9:50
Distance      : 0.3897


STEP 4: Cross-encoder reranking
Reranker detected.
Computing rerank scores...
Chunk 9:50:0 | Original Rank: 1 | Rerank Score: 0.9980

Reranking completed.

NEW ORDER AFTER RERANKING:
--------------------------------------------------------------------------------
[NEW RANK 1] 9:50:0 | Original Rank=1 | Score=0.9980

STEP 5: Diversity filtering
Chunks after diversity filtering: 1

Kept chunk 9:50:0 | Parent=9:50

STEP 6: Selecting final chunks
Final chunk count: 1

[FINAL 1]
Chunk ID      : 9:50:0
Parent ID    

...


DONE
GENERATION COMPLETED
STEP 1: RETRIEVING CONTEXT
STEP 1: Loading QuranDoc store
Docs path: ../../data/gold/quran_docs.pkl
Successfully loaded 10 QuranDoc objects.

STEP 2: Dense semantic retrieval
Query: What does the tafsir of Surah 9:8 say about the idolators' covenant?
Top-K retrieval: 1
Vector search completed successfully.

STEP 3: Building retrieved chunk candidates
[1]
Chunk ID      : 43:19:0
Parent ID     : 43:19
Distance      : 0.4626


STEP 4: Cross-encoder reranking
Reranker detected.
Computing rerank scores...
Chunk 43:19:0 | Original Rank: 1 | Rerank Score: 0.1664

Reranking completed.

NEW ORDER AFTER RERANKING:
--------------------------------------------------------------------------------
[NEW RANK 1] 43:19:0 | Original Rank=1 | Score=0.1664

STEP 5: Diversity filtering
Chunks after diversity filtering: 1

Kept chunk 43:19:0 | Parent=43:19

STEP 6: Selecting final chunks
Final chunk count: 1

[FINAL 1]
Chunk ID      : 43:19:0
Parent ID     : 43:19
Original Rank : 

 99%|█████████▊| 72/73 [2:18:48<02:35, 155.95s/it]


DONE
GENERATION COMPLETED
STEP 1: RETRIEVING CONTEXT
STEP 1: Loading QuranDoc store
Docs path: ../../data/gold/quran_docs.pkl
Successfully loaded 10 QuranDoc objects.

STEP 2: Dense semantic retrieval
Query: What does the tafsir of Surah 90:19 say about those who disbelieve in Allah's signs?
Top-K retrieval: 1
Vector search completed successfully.

STEP 3: Building retrieved chunk candidates
[1]
Chunk ID      : 2:121:31
Parent ID     : 2:121
Distance      : 0.3749


STEP 4: Cross-encoder reranking
Reranker detected.
Computing rerank scores...
Chunk 2:121:31 | Original Rank: 1 | Rerank Score: 0.2513

Reranking completed.

NEW ORDER AFTER RERANKING:
--------------------------------------------------------------------------------
[NEW RANK 1] 2:121:31 | Original Rank=1 | Score=0.2513

STEP 5: Diversity filtering
Chunks after diversity filtering: 1

Kept chunk 2:121:31 | Parent=2:121

STEP 6: Selecting final chunks
Final chunk count: 1

[FINAL 1]
Chunk ID      : 2:121:31
Parent ID     : 2

100%|██████████| 73/73 [2:20:52<00:00, 115.78s/it]


DONE
GENERATION COMPLETED


In [12]:
from rouge_metric import PyRouge
rouge = PyRouge( rouge_l=True)

In [13]:
## Read data
data_eval=pd.read_json('../../data/evaluation/eval_generator_dataset_full.jsonl',lines=True)


In [14]:
rouge = PyRouge( rouge_l=True)

In [15]:
rouge_data = []
for i,row in data_eval.iterrows():
    print(f'processing row {i}')
    score = rouge.evaluate(
        [str(row['generated_answer'])],[[str(row['ground_truth_answer'])]]
    )
    rouge_data.append(score['rouge-l'])
    

processing row 0
processing row 1
processing row 2
processing row 3
processing row 4
processing row 5
processing row 6
processing row 7
processing row 8
processing row 9
processing row 10
processing row 11
processing row 12
processing row 13
processing row 14
processing row 15
processing row 16
processing row 17
processing row 18
processing row 19
processing row 20
processing row 21
processing row 22
processing row 23
processing row 24
processing row 25
processing row 26
processing row 27
processing row 28
processing row 29
processing row 30
processing row 31
processing row 32
processing row 33
processing row 34
processing row 35
processing row 36
processing row 37
processing row 38
processing row 39
processing row 40
processing row 41
processing row 42
processing row 43
processing row 44
processing row 45
processing row 46
processing row 47
processing row 48
processing row 49
processing row 50
processing row 51
processing row 52
processing row 53
processing row 54
processing row 55
pr

In [16]:
rouge_evaluation = pd.DataFrame(
    rouge_data
)

In [17]:
rouge_evaluation

,r,p,f
0,0.409091,0.126761,0.193548
1,0.875000,0.333333,0.482759
2,0.078431,0.090909,0.084211
3,0.206897,0.240000,0.222222
4,0.565217,0.110169,0.184397
...,...,...,...
68,0.333333,0.046053,0.080925
69,0.094340,0.092593,0.093458
70,0.187500,0.047872,0.076271
71,0.314815,0.089005,0.138776


In [19]:
import plotly.graph_objects as go

fig = go.Figure()

# =========================
# Main ROUGE curves
# =========================
fig.add_trace(
    go.Scatter(
        x=rouge_evaluation.index,
        y=rouge_evaluation['r'] * 100,
        mode="lines+markers",
        name="ROUGE-L Recall",
        line=dict(width=3),
        marker=dict(size=7),
    )
)

fig.add_trace(
    go.Scatter(
        x=rouge_evaluation.index,
        y=rouge_evaluation['p'] * 100,
        mode="lines+markers",
        name="ROUGE-L Precision",
        line=dict(width=3),
        marker=dict(size=7),
    )
)

fig.add_trace(
    go.Scatter(
        x=rouge_evaluation.index,
        y=rouge_evaluation['f'] * 100,
        mode="lines+markers",
        name="ROUGE-L F1",
        line=dict(width=4),
        marker=dict(size=8),
    )
)

# =========================
# Mean lines
# =========================
mean_f1 = rouge_evaluation['f'].mean() * 100
mean_p = rouge_evaluation['p'].mean() * 100
mean_r = rouge_evaluation['r'].mean() * 100

fig.add_hline(
    y=mean_f1,
    line_width=3,
    line_dash="dash",
    annotation_text=f"Mean F1 = {mean_f1:.2f}%",
    annotation_position="top left"
)

fig.add_hline(
    y=mean_p,
    line_width=2,
    line_dash="dot",
    annotation_text=f"Mean Precision = {mean_p:.2f}%",
    annotation_position="bottom left"
)

fig.add_hline(
    y=mean_r,
    line_width=2,
    line_dash="dot",
    annotation_text=f"Mean Recall = {mean_r:.2f}%",
    annotation_position="bottom right"
)


# =========================
# Layout
# =========================
fig.update_layout(
    title={
        "text": "ROUGE-L Evaluation of the Quran RAG Generator",
        "x": 0.5,
        "xanchor": "center",
        "font": dict(size=24)
    },
    xaxis_title="Evaluation Samples",
    yaxis_title="Score (%)",
    template="plotly_white",
    
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    font=dict(size=15)
)

# Better axes
fig.update_xaxes(
    showgrid=False,
    zeroline=False
)

fig.update_yaxes(
    range=[0, 100],
    showgrid=True,
    gridwidth=1
)

fig.show()

In [ ]:
def print_case(title, df, data_eval):
    print("\n" + "="*80)
    print(title)
    print("="*80)

    for i, (idx, row) in enumerate(df.iterrows()):
        original = data_eval.loc[idx]

        print(f"\n📌 Case {i+1} | Index: {idx}")
        print("-"*80)

        print("\n🔹 Ground Truth Answer:")
        print(original['ground_truth_answer'])

        print("\n🔸 Generated Answer:")
        print(original['generated_answer'])

        print("\n📊 Scores:")
        print(f"Precision: {row['p']:.3f}")
        print(f"Recall:    {row['r']:.3f}")
        print(f"F1:        {row['f']:.3f}")

In [ ]:
# =========================
# LOW PRECISION (Top 3 worst)
# =========================
low_precision = rouge_evaluation.nsmallest(3, 'p')

# =========================
# LOW RECALL (Top 3 worst)
# =========================
low_recall = rouge_evaluation.nsmallest(3, 'r')

# =========================
# HIGH PRECISION (Top 3 best)
# =========================
high_precision = rouge_evaluation.nlargest(3, 'p')

# =========================
# HIGH RECALL (Top 3 best)
# =========================
high_recall = rouge_evaluation.nlargest(3, 'r')

In [ ]:
print_case("🔴 LOW PRECISION CASES (Hallucination / verbosity issues)", low_precision, data_eval)

print_case("🔵 LOW RECALL CASES (Retrieval failure)", low_recall, data_eval)

print_case("🟢 HIGH PRECISION CASES (Good conciseness)", high_precision, data_eval)

print_case("🟣 HIGH RECALL CASES (Good coverage)", high_recall, data_eval)


🔴 LOW PRECISION CASES (Hallucination / verbosity issues)

📌 Case 1 | Index: 22
--------------------------------------------------------------------------------

🔹 Ground Truth Answer:
The three steps are: (1) admonish/advise her, (2) abandon her in bed, and (3) beat her (lightly, without severe beating).

🔸 Generated Answer:
Direct Answer: The Quran does not explicitly mention specific steps for a husband to take when facing his wife's ill-conduct (nushuz). However, Surah 4:34 provides guidance on the husband's right to chastise his disobedient wife.

Quran References: Surah 4:34

Tafsir Insights: In Surah 4:34, it is mentioned that a disobedient wife may be admonished, then left in bed (separated) before being beaten, but only lightly. It is important to note that the verses emphasize the necessity of fairness and justice in such situations.

1. Admonish her: The husband should initially admonish his disobedient wife and encourage her to return to righteous behavior.
2. Separate from

## BERTSCORE

In [22]:
from bert_score import BERTScorer

# Prepare data
candidates = data_eval['generated_answer'].astype(str).tolist()
references = data_eval['ground_truth_answer'].astype(str).tolist()

# Initialize scorer (better model)
scorer = BERTScorer(model_type='roberta-large')

# Compute scores
P, R, F1 = scorer.score(candidates, references)

print(f"BERTScore Precision: {P.mean().item():.4f}")
print(f"BERTScore Recall:    {R.mean().item():.4f}")
print(f"BERTScore F1:        {F1.mean().item():.4f}")

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 5740.81it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore Precision: 0.8170
BERTScore Recall:    0.8592
BERTScore F1:        0.8371


In [23]:
import plotly.graph_objects as go

bert_evaluation = pd.DataFrame({
    "p": P.numpy(),
    "r": R.numpy(),
    "f": F1.numpy()
})
fig = go.Figure()

# =========================
# Main BERTScore curves
# =========================
fig.add_trace(
    go.Scatter(
        x=bert_evaluation.index,
        y=bert_evaluation['r'] * 100,
        mode="lines+markers",
        name="BERTScore Recall",
        line=dict(width=3),
        marker=dict(size=7),
    )
)

fig.add_trace(
    go.Scatter(
        x=bert_evaluation.index,
        y=bert_evaluation['p'] * 100,
        mode="lines+markers",
        name="BERTScore Precision",
        line=dict(width=3),
        marker=dict(size=7),
    )
)

fig.add_trace(
    go.Scatter(
        x=bert_evaluation.index,
        y=bert_evaluation['f'] * 100,
        mode="lines+markers",
        name="BERTScore F1",
        line=dict(width=4),
        marker=dict(size=8),
    )
)

# =========================
# Mean lines
# =========================
mean_f1 = bert_evaluation['f'].mean() * 100
mean_p = bert_evaluation['p'].mean() * 100
mean_r = bert_evaluation['r'].mean() * 100

fig.add_hline(
    y=mean_f1,
    line_width=3,
    line_dash="dash",
    annotation_text=f"Mean F1 = {mean_f1:.2f}%",
    annotation_position="top left"
)

fig.add_hline(
    y=mean_p,
    line_width=2,
    line_dash="dot",
    annotation_text=f"Mean Precision = {mean_p:.2f}%",
    annotation_position="bottom left"
)

fig.add_hline(
    y=mean_r,
    line_width=2,
    line_dash="dot",
    annotation_text=f"Mean Recall = {mean_r:.2f}%",
    annotation_position="bottom right"
)

# =========================
# Layout
# =========================
fig.update_layout(
    title={
        "text": "BERTScore Evaluation of the Quran RAG Generator",
        "x": 0.5,
        "xanchor": "center",
        "font": dict(size=24)
    },
    xaxis_title="Evaluation Samples",
    yaxis_title="Score (%)",
    template="plotly_white",
    
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    font=dict(size=15)
)

# =========================
# Axes styling
# =========================
fig.update_xaxes(
    showgrid=False,
    zeroline=False
)

fig.update_yaxes(
    range=[0, 100],
    showgrid=True,
    gridwidth=1
)

fig.show()

In [25]:
best_precision = bert_evaluation.maxind("p")

AttributeError: 'DataFrame' object has no attribute 'maxind'